In [ ]:
# Hopsworks-Projektverbindungsskript
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

# Sicheres Laden der .env-Datei aus dem Projektstamm
project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

# Umgebungsvariablen abrufen
api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")

if not api_key or not project_name:
    raise ValueError(
        "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
    )

# Verbindung zum Hopsworks-Projekt herstellen
project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)

print(f"✅ Erfolgreich verbunden mit Projekt: {project.name}")

In [ ]:
# Rohdaten via API abrufen
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(latitude: float, longitude: float, past_days: int = 7, forecast_days: int = 1) -> pd.DataFrame:
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "surface_pressure",
            "cloud_cover",
            "wind_speed_10m",
        ],
        "past_days": past_days,
        "forecast_days": forecast_days,
        "timezone": "Europe/Berlin",
    }

    response = openmeteo.weather_api(url, params=params)[0]
    hourly = response.Hourly()
    start_ts = hourly.Time()
    num_points = len(hourly.Variables(0).ValuesAsNumpy())
    start_dt = pd.to_datetime(start_ts, unit="s", utc=True).tz_convert("Europe/Berlin")
    dates = pd.date_range(start=start_dt, periods=num_points, freq="h")

    df = pd.DataFrame({
        "time": dates,
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "precipitation": hourly.Variables(2).ValuesAsNumpy(),
        "pressure_msl": hourly.Variables(3).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
        "cloud_cover": hourly.Variables(5).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(6).ValuesAsNumpy(),
    })

    df["latitude"] = latitude
    df["longitude"] = longitude
    df["location_id"] = f"{latitude}_{longitude}"
    return df


# Beispiel: Standorte abrufen (z.B. Städte mit Unwetterrisiko)
locations = [
    {"name": "Muenchen", "lat": 48.1351, "lon": 11.5820},
    {"name": "Hamburg", "lat": 53.5511, "lon": 9.9937},
]

raw_dfs = []
for loc in locations:
    df = fetch_weather_data(loc["lat"], loc["lon"])
    df["location_name"] = loc["name"]
    raw_dfs.append(df)

raw_df = pd.concat(raw_dfs, ignore_index=True)
print(f"✅ {len(raw_df)} Zeilen Rohdaten abgerufen")
print(raw_df.head())

In [ ]:
# Features engineeren (Rolling Windows, Aggregationen)
def engineer_features(df):
    """
    Erstellt Features, die für Unwetter-Prognosen entscheidend sind:
    - Rolling Windows für Trends
    - Druckabfall (Vorbote für Stürme)
    - Windböen-Anomalien
    """
    df = df.sort_values(["location_id", "time"]).copy()
    df["pressure_msl"] = df.get("pressure_msl", df.get("surface_pressure", pd.Series(0.0, index=df.index)))
    df["wind_gusts_10m"] = df.get("wind_gusts_10m", df.get("wind_speed_10m", pd.Series(0.0, index=df.index)))
    df["cape"] = pd.to_numeric(
        df.get("cape", pd.Series(0.0, index=df.index)),
        errors="coerce",
    ).fillna(0.0)

    grouped = df.groupby("location_id")

    # 🌀 Rolling Windows (z.B. 3h, 6h, 12h)
    for window in [3, 6, 12]:
        df[f"precip_rolling_sum_{window}h"] = grouped["precipitation"].transform(
            lambda x: x.rolling(window, min_periods=1).sum()
        )
        df[f"wind_gust_max_{window}h"] = grouped["wind_gusts_10m"].transform(
            lambda x: x.rolling(window, min_periods=1).max()
        )
        df[f"pressure_mean_{window}h"] = grouped["pressure_msl"].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )

    # Druckabfall als Sturm-Indikator (kritisch!)
    df["pressure_change_3h"] = grouped["pressure_msl"].transform(
        lambda x: x.diff(periods=3)
    )
    df["pressure_drop_rate"] = df["pressure_change_3h"] / 3

    # Windböen-Anomalie (Abweichung vom Mittelwert)
    df["wind_gust_anomaly"] = grouped["wind_gusts_10m"].transform(
        lambda x: x - x.rolling(24, min_periods=1).mean()
    )

    # Temperaturdifferenz (Fronten-Indikator)
    df["temp_change_3h"] = grouped["temperature_2m"].transform(
        lambda x: x.diff(periods=3)
    )

    # CAPE-Kategorien (Gewitterpotential)
    df["cape_risk_level"] = pd.cut(
        df["cape"], bins=[0, 500, 1500, 3000, float("inf")],
        labels=["niedrig", "moderat", "hoch", "extrem"]
    )

    # Label erstellen: Unwetter-Trigger (Beispiel-Logik)
    df["is_severe_weather"] = (
        (df["wind_gusts_10m"] > 60) |
        (df["precip_rolling_sum_3h"] > 20) |
        (df["pressure_drop_rate"] < -2)
    ).astype(int)

    return df

feature_df = engineer_features(raw_df)
print(feature_df.head())
print(f"✅ Features erstellt: {feature_df.shape[1]} Spalten")

In [ ]:
# Access the feature store
fs = project.get_feature_store()